## RQ8: How does a high density off gas stations impact the fuel prices?

In [ ]:
import glob
import sys
from pathlib import Path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

from sklearn.cluster import DBSCAN

from scripts.rq_8_competition_script import (
    analyse_motorway_clusters,
    compute_cluster_counts_over_time,
    create_csv_with_cluster_labels,
    get_csv_data,
    join_labels_and_group,
    perform_dbscan,
    plot_cluster_counts_over_time,
    plot_cluster_difference,
    plot_cluster_prices,
    plot_clusters,
    plot_motorway_cluster_pies,
    plot_yearly_boxplot,
    save_png,
)
from scripts.rq8_build_station_daily_by_month import (
    build_station_daily_by_month,
)

In [ ]:
# Define all input and output paths here.

INPUT_ROOT = "C:\\Users\\Bjarne\\Desktop\\Uni\\Data Science Projekt\\PersonalTesting\\dbscan"
OUTPUT_ROOT = "C:\\Users\\Bjarne\\Desktop\\Uni\\Data Science Projekt\\PersonalTesting\\dbscan\\labeled_stations"
INPUT_PRICES = "C:\\Users\\Bjarne\\Desktop\\Uni\\Data Science Projekt\\TankKoenigData"
OUTPUT_PRICES = "C:\\Users\\Bjarne\\Desktop\\Uni\\Data Science Projekt\\PersonalTesting\\dbscan"

In [ ]:
#read the data from the CSV file
df = get_csv_data(INPUT_ROOT + "\\stations.csv")

In [ ]:
# Here we tune/set our parameters for DBSCAN.

eps1 = 2 #in km
min_samples1 = 4

eps2 = 0.2 #in km
min_samples2 = 2

# We now execute DBSCAN and plot the results.
labels1 = perform_dbscan(df, eps1, min_samples1)
figure1 = plot_clusters(df, labels1)
figure1.show()

labels2 = perform_dbscan(df, eps2, min_samples2)
figure2 = plot_clusters(df, labels2)
save_png(figure2, "dbscan_clusters_eps.png", True)

figure2.show()

Number of clusters: 799
[798  -1   0 ...  -1  -1 536]


Number of clusters: 1138
[  -1   -1   -1 ...  449 1020 1100]


### Now we will create all necessary files for a more efficient analyse afterwards.

In [ ]:
clusters = pl.read_csv(
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps1}_min_samples{min_samples1}.csv"
)

noise_count_esp1 = clusters.filter(pl.col("cluster") == -1).height

clusters = pl.read_csv(
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps2}_min_samples{min_samples2}.csv"
)

noise_count_esp2 = clusters.filter(pl.col("cluster") == -1).height

print("Noise stations Cluster 1:", noise_count_esp1)
print("Noise stations Cluster 2:", noise_count_esp2)

Noise stations Cluster 1: 7631
Noise stations Cluster 2: 13038


In [ ]:
create_csv_with_cluster_labels(
    df,    
    labels1, 
    OUTPUT_ROOT + "\\stations_clusters" + f"_eps{eps1}_min_samples{min_samples1}.csv"
    )

create_csv_with_cluster_labels(
    df, 
    labels2, 
    OUTPUT_ROOT + "\\stations_clusters" + f"_eps{eps2}_min_samples{min_samples2}.csv"
    )

In [ ]:
# Call the build_station_daily_by_month function with proper parameters
# INPUT_ROOT should contain prices/{year}/{month}/ subdirectories
build_station_daily_by_month(
    data_root=INPUT_PRICES,
    derived_root=OUTPUT_PRICES,
    start_year=2014,
    end_year=2026
)

[2014-01] reading: C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\TankKoenigData\prices\2014\01\*-prices.csv
[2014-01] no files / unreadable -> skip
[2014-02] reading: C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\TankKoenigData\prices\2014\02\*-prices.csv
[2014-02] no files / unreadable -> skip
[2014-03] reading: C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\TankKoenigData\prices\2014\03\*-prices.csv
[2014-03] no files / unreadable -> skip
[2014-04] reading: C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\TankKoenigData\prices\2014\04\*-prices.csv
[2014-04] no files / unreadable -> skip
[2014-05] reading: C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\TankKoenigData\prices\2014\05\*-prices.csv
[2014-05] no files / unreadable -> skip
Skip (exists): C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\PersonalTesting\dbscan\station_daily_mean_and_median_by_month\2014\2014-06.parquet
Skip (exists): C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\PersonalTesting\dbscan\station_dai

In [ ]:
# Displays a sample of the resulting parquet files to verify the structure and content.

parquet_dir = Path(OUTPUT_PRICES) / "station_daily_mean_and_median_by_month"
parquet_files = sorted(glob.glob(str(parquet_dir / "**" / "*.parquet"), recursive=True))

if parquet_files:
    df_sample = pl.read_parquet(parquet_files[0])
    print(f"Table structure from: {parquet_files[0]}\n")
    print(df_sample.head(10))
    print(f"\nColumns: {df_sample.columns}")
    print(f"Shape: {df_sample.shape}")
else:
    print("No parquet files found. Run the previous cell first.")

Table structure from: C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\PersonalTesting\dbscan\station_daily_mean_and_median_by_month\2014\2014-06.parquet

shape: (10, 9)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬──────────┬───────────┬──────────┐
│ station_uu ┆ day       ┆ diesel_me ┆ diesel_me ┆ … ┆ e5_median ┆ e10_mean ┆ e10_media ┆ n_events │
│ id         ┆ ---       ┆ an        ┆ dian      ┆   ┆ ---       ┆ ---      ┆ n         ┆ ---      │
│ ---        ┆ date      ┆ ---       ┆ ---       ┆   ┆ f64       ┆ f64      ┆ ---       ┆ u32      │
│ str        ┆           ┆ f64       ┆ f64       ┆   ┆           ┆          ┆ f64       ┆          │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪══════════╪═══════════╪══════════╡
│ 00041450-0 ┆ 2014-06-0 ┆ 1.318     ┆ 1.318     ┆ … ┆ 1.538     ┆ 1.498    ┆ 1.498     ┆ 1        │
│ 002-4444-8 ┆ 8         ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ 888-acdc00 ┆        

# Now we will start plotting and analyzing the Data

In [ ]:
# Choose the Fuel Type to analyze, from: diesel, e5, e10
FUEL_TYPE = "e5"

In [ ]:
parquet_dir = Path(OUTPUT_PRICES) / "station_daily_mean_and_median_by_month"
parquet_files = sorted(glob.glob(str(parquet_dir / "**" / "*.parquet"), recursive=True))


# Diesel, Cluster Set 1
fig_1 = plot_cluster_prices(
    parquet_files,
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps1}_min_samples{min_samples1}.csv",
    fuel = FUEL_TYPE,
    title = FUEL_TYPE + "Prices - Cluster Set 1"
)
fig_1.show()

fig_2 = plot_cluster_prices(
    parquet_files,
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps2}_min_samples{min_samples2}.csv",
    fuel = FUEL_TYPE,
    title = FUEL_TYPE + "Prices - Cluster Set 2"
)
fig_2.show()

### First we will start start cleaning all the Motorwaystations from the Dataset, to prevent scewed results.

In [ ]:
motorway_stations = (
    pl.read_csv(
        INPUT_ROOT + "\\autobahn_stations.csv",
        null_values=["nicht", "NA", "N/A", "", "Nicht"]
    )
    .select(pl.col("uuid").alias("station_uuid"))
)

# Plot the distribution of motorway stations across clusters for both parameter sets.

fig = plot_motorway_cluster_pies(
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps1}_min_samples{min_samples1}.csv",
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps2}_min_samples{min_samples2}.csv",
    motorway_stations
)

fig.show()

Now we will plot and compare the cleaned data from both cluster sets.

In [ ]:
fig_diesel_clean_esp1 = plot_cluster_prices(
    parquet_files,
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps1}_min_samples{min_samples1}.csv",
    fuel = FUEL_TYPE,
    motorway_df=motorway_stations,
    title = FUEL_TYPE + "Prices - Cluster Set 1 (no motorway stations)"
) 

fig_diesel_clean_esp2 = plot_cluster_prices(
    parquet_files,
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps2}_min_samples{min_samples2}.csv",
    fuel = FUEL_TYPE,
    motorway_df=motorway_stations,
    title = FUEL_TYPE + "Prices - Cluster Set 2 (no motorway stations)"
)   



fig_diesel_clean_esp1.show()
fig_diesel_clean_esp2.show()

For **easier visual understanding** we will now only plot the **difference** between the clustered und unclustered data. 
**Positive** numbers indicate a **higher price from the clustered data**, for example: 0.30 ==> The clustered Stations costs 30 cents more. Negative Numbers on the other hand indicate that clustered stations are cheaper, than the unclustered.

In [ ]:
parquet_dir = Path(OUTPUT_PRICES) / "station_daily_mean_and_median_by_month"
parquet_files = sorted(glob.glob(str(parquet_dir / "**" / "*.parquet"), recursive=True))

fig_diff, diff_df1 = plot_cluster_difference(
    parquet_files,
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps1}_min_samples{min_samples1}.csv",
    fuel = FUEL_TYPE,
    motorway_df=motorway_stations,
    title=FUEL_TYPE + " Cluster 1 vs Noise Difference (Mean & Median, no motorway)"
)
fig_diff.show()

diff_df1.to_csv(OUTPUT_ROOT + f"\\cluster_1_diff_{FUEL_TYPE}.csv", index=False)

fig_diff, diff_df2 = plot_cluster_difference(
    parquet_files,
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps2}_min_samples{min_samples2}.csv",
    fuel = FUEL_TYPE,
    motorway_df=motorway_stations,
    title = FUEL_TYPE + " Cluster 2 vs Noise Difference (Mean & Median, no motorway)"
)
fig_diff.show()

diff_df2.to_csv(OUTPUT_ROOT + f"\\cluster_2_diff_{FUEL_TYPE}.csv", index=False)

### Boxplot
Because of the high oscillation, the graph is quite cluttered.
To easier make out underlying trends we will now use a boxplot to visualize the price difference.

In [ ]:
fig_cluster1 = plot_yearly_boxplot(diff_df1, "Cluster Set 1", FUEL_TYPE)
fig_cluster1.show()

fig_cluster2 = plot_yearly_boxplot(diff_df2, "Cluster Set 2", FUEL_TYPE)
save_png(fig_cluster2, OUTPUT_ROOT + f"\\cluster_difference_yearly_boxplot_eps{eps2}_min_samples{min_samples2}.png")
fig_cluster2.show()

While no significant differences can be observed in the first clustering approach,  
a clear upward trend starting in **2023** becomes evident.

This pattern is **not present to the same extent** in the first clustering,  
which allows us to largely **rule out effects specific to urban areas**.

Although a higher station density may indicate a more profitable environment,  
This alone does not sufficiently explain the sudden price increase from 2023 onward.

To rule out an imbalance of our data over time we will now plot the number of
clustered and unclustered Stations over the whole timeframe.

In [ ]:
daily_prices = pl.concat([pl.read_parquet(f) for f in parquet_files])
counts_eps1 = compute_cluster_counts_over_time(
    daily_prices, 
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps1}_min_samples{min_samples1}.csv"
)
counts_eps2 = compute_cluster_counts_over_time(
    daily_prices, 
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps2}_min_samples{min_samples2}.csv"
)


In [ ]:
fig_counts_eps1 = plot_cluster_counts_over_time(counts_eps1, title="Cluster Set 1: Cluster vs Noise")
fig_counts_eps1.show()

fig_counts_eps2 = plot_cluster_counts_over_time(counts_eps2, title="Cluster Set 2: Cluster vs Noise")
fig_counts_eps2.show()

## Conclusion

There is no clear trend in the he share of clustered stations over time, especially not one that would explain the difference from 2023 onwards. Although we can assume a local enviroment of competition where the demand of fuel can support atleast two stations, **this does not explain the Sharp increase by 2023**. Until the end of the project we could not isolate an effect, that would explain the observed behavior. The only shared characteristic of these stations, beside the assumed higher demand, is the proximity to another Station. 

At the current time we **cannot provide a conclusive explanation for the observed behavior**.